In [29]:
import pandas as pd
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder

# ==========================================
# 1. Download NLTK Resources
# ==========================================

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


# ==========================================
# 2. Load Dataset
# ==========================================

df = pd.read_csv("emotions.csv")

# Rename label column to emotions
df = df.rename(columns={"label": "emotions"})

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
print(df.head())

print("\nMissing Values:")
print(df.isnull().sum())


# ==========================================
# 3. Keep Original Text
# ==========================================

df["original_text"] = df["text"]


# ==========================================
# 4. Encode Emotion Labels
# ==========================================

label_encoder = LabelEncoder()

df["emotion_encoded"] = label_encoder.fit_transform(df["emotions"])


# Display Label Mapping
emotion_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

print("\nEmotion Mapping:")
print(emotion_mapping)


# ==========================================
# 5. Load English Stopwords
# ==========================================

stop_words = set(stopwords.words("english"))


# ==========================================
# 6. Complete Text Cleaning Function
# ==========================================

def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    translator = str.maketrans("", "", string.punctuation)
    text = text.translate(translator)

    # Remove numbers
    translator = str.maketrans("", "", string.digits)
    text = text.translate(translator)

    # Remove emojis and special characters
    text = text.encode("ascii", "ignore").decode("ascii")

    # Tokenization
    words = word_tokenize(text)

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Join words back into text
    return " ".join(words)


# ==========================================
# 7. Apply Cleaning
# ==========================================

df["cleaned_text"] = df["original_text"].apply(clean_text)


# ==========================================
# 8. Display Original vs Cleaned Text
# ==========================================

print("\nOriginal vs Cleaned Text:")
print(df[["original_text", "cleaned_text"]].head(10))


# ==========================================
# 9. Save Cleaned Dataset
# ==========================================

df.to_csv("cleaned_emotions.csv", index=False)

print("\n====================================")
print("Cleaned dataset saved successfully!")
print("File: cleaned_emotions.csv")
print("====================================")


# ==========================================
# 10. Emotion Counts
# ==========================================

print("\nEmotion Counts:")
print(df["emotions"].value_counts())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset Shape: (416809, 2)

First 5 Rows:
                                                text  emotions
0      i just feel really helpless and heavy hearted         4
1  ive enjoyed being able to slouch about relax a...         0
2  i gave up my internship with the dmrg and am f...         4
3                         i dont know i feel so lost         0
4  i am a kindergarten teacher and i am thoroughl...         4

Missing Values:
text        0
emotions    0
dtype: int64

Emotion Mapping:
{np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5): np.int64(5)}

Original vs Cleaned Text:
                                       original_text  \
0      i just feel really helpless and heavy hearted   
1  ive enjoyed being able to slouch about relax a...   
2  i gave up my internship with the dmrg and am f...   
3                         i dont know i feel so lost   
4  i am a kindergarten teacher and i am 

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("cleaned_emotions.csv")

X = df["cleaned_text"]
y = df["emotions"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (333447,)
X_test shape : (83362,)
y_train shape: (333447,)
y_test shape : (83362,)


In [32]:
# Check missing values
print("Missing values in X_train:", X_train.isnull().sum())
print("Missing values in X_test:", X_test.isnull().sum())

Missing values in X_train: 13
Missing values in X_test: 3


In [33]:
# Replace missing text with an empty string
X_train = X_train.fillna("")
X_test = X_test.fillna("")

In [34]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train)

X_test_bow = vectorizer.transform(X_test)

print("X_train_bow shape:", X_train_bow.shape)
print("X_test_bow shape :", X_test_bow.shape)

print("\nFirst 20 Feature Names:")
print(vectorizer.get_feature_names_out()[:20])

X_train_bow shape: (333447, 67774)
X_test_bow shape : (83362, 67774)

First 20 Feature Names:
['aa' 'aaa' 'aaaa' 'aaaaaaaaaaaaaaaaggghhhh' 'aaaaaaaall' 'aaaaaand'
 'aaaaah' 'aaaaahhhhhh' 'aaaaall' 'aaaaand' 'aaaand' 'aaah' 'aaahs'
 'aaand' 'aaargh' 'aaawesome' 'aab' 'aabsolutely' 'aac' 'aacc']


In [35]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model = MultinomialNB()

nb_model.fit(X_train_bow, y_train)

y_pred_bow = nb_model.predict(X_test_bow)

accuracy_bow = accuracy_score(y_test, y_pred_bow)

print("Bag of Words + MultinomialNB Accuracy:", accuracy_bow)

Bag of Words + MultinomialNB Accuracy: 0.8676375326887551


In [36]:
vocabulary = vectorizer.get_feature_names_out()

print("Total Vocabulary Size:", len(vocabulary))

print("\n15 Vocabulary Words:")
print(vocabulary[:15])

sample_document = X_train.iloc[0]

print("\nSample Document:")
print(sample_document)

sample_bow = vectorizer.transform([sample_document])

print("\nBoW Vector:")
print(sample_bow.toarray())

Total Vocabulary Size: 67774

15 Vocabulary Words:
['aa' 'aaa' 'aaaa' 'aaaaaaaaaaaaaaaaggghhhh' 'aaaaaaaall' 'aaaaaand'
 'aaaaah' 'aaaaahhhhhh' 'aaaaall' 'aaaaand' 'aaaand' 'aaah' 'aaahs'
 'aaand' 'aaargh']

Sample Document:
ive blabbed enough tonight im tired ive feeling pretty crappy kentucky weather

BoW Vector:
[[0 0 0 ... 0 0 0]]


In [37]:
from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))

X_train_bigram = bigram_vectorizer.fit_transform(X_train)

X_test_bigram = bigram_vectorizer.transform(X_test)

print("X_train_bigram shape:", X_train_bigram.shape)
print("X_test_bigram shape :", X_test_bigram.shape)

print("\nFirst 20 Bigram Features:")
print(bigram_vectorizer.get_feature_names_out()[:20])

X_train_bigram shape: (333447, 1273457)
X_test_bigram shape : (83362, 1273457)

First 20 Bigram Features:
['aa' 'aa button' 'aa config' 'aa didnt' 'aa feel' 'aa finally' 'aa full'
 'aa linkurl' 'aa meeting' 'aa order' 'aa raha' 'aa yesterdays' 'aaa'
 'aaa taste' 'aaa team' 'aaaa' 'aaaa whatever' 'aaaaaaaaaaaaaaaaggghhhh'
 'aaaaaaaaaaaaaaaaggghhhh three' 'aaaaaaaall']


In [38]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_bigram = MultinomialNB()

nb_bigram.fit(X_train_bigram, y_train)

y_pred_bigram = nb_bigram.predict(X_test_bigram)

accuracy_bigram = accuracy_score(y_test, y_pred_bigram)

print("Unigram (Basic BoW) Accuracy:", accuracy_bow)
print("Bigram BoW Accuracy:", accuracy_bigram)

if accuracy_bigram > accuracy_bow:
    print("\nBigram BoW performed better.")
elif accuracy_bigram < accuracy_bow:
    print("\nUnigram BoW performed better.")
else:
    print("\nBoth methods have the same accuracy.")

Unigram (Basic BoW) Accuracy: 0.8676375326887551
Bigram BoW Accuracy: 0.8153955039466424

Unigram BoW performed better.


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape :", X_test_tfidf.shape)

print("\nFirst 15 TF-IDF Feature Names:")
print(tfidf_vectorizer.get_feature_names_out()[:15])

X_train_tfidf shape: (333447, 67774)
X_test_tfidf shape : (83362, 67774)

First 15 TF-IDF Feature Names:
['aa' 'aaa' 'aaaa' 'aaaaaaaaaaaaaaaaggghhhh' 'aaaaaaaall' 'aaaaaand'
 'aaaaah' 'aaaaahhhhhh' 'aaaaall' 'aaaaand' 'aaaand' 'aaah' 'aaahs'
 'aaand' 'aaargh']


In [40]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_tfidf = MultinomialNB()

nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("TF-IDF + MultinomialNB Accuracy:", accuracy_tfidf)

TF-IDF + MultinomialNB Accuracy: 0.7645809841414554


In [43]:
comparison = pd.DataFrame({
    "Vectorization Method": [
        "Bag of Words (Unigrams)",
        "Bag of Words (Unigrams + Bigrams)",
        "TF-IDF"
    ],
    "Accuracy": [
        accuracy_bow,
        accuracy_bigram,
        accuracy_tfidf
    ]
})

print(comparison)

best_index = comparison["Accuracy"].idxmax()
best_method = comparison.loc[best_index, "Vectorization Method"]
best_accuracy = comparison.loc[best_index, "Accuracy"]

print("\nBest Performing Method:", best_method)
print("Best Accuracy:", best_accuracy)

print("\nObservation:")
print(
    f"{best_method} performed the best with an accuracy of "
    f"{best_accuracy:.4f}. "
    "\nThe performance depends on how effectively the vectorizer \n"
    "represents important information in the text."
)

                Vectorization Method  Accuracy
0            Bag of Words (Unigrams)  0.867638
1  Bag of Words (Unigrams + Bigrams)  0.815396
2                             TF-IDF  0.764581

Best Performing Method: Bag of Words (Unigrams)
Best Accuracy: 0.8676375326887551

Observation:
Bag of Words (Unigrams) performed the best with an accuracy of 0.8676. 
The performance depends on how effectively the vectorizer 
represents important information in the text.


In [44]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

df = pd.read_csv("cleaned_emotions.csv")

X = df["cleaned_text"]
y = df["emotions"]

X = X.fillna("")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

count_vectorizer = CountVectorizer()

X_train_bow = count_vectorizer.fit_transform(X_train)
X_test_bow = count_vectorizer.transform(X_test)


bow_model = MultinomialNB()

bow_model.fit(X_train_bow, y_train)

y_pred_bow = bow_model.predict(X_test_bow)

bow_accuracy = accuracy_score(y_test, y_pred_bow)


tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


tfidf_model = MultinomialNB()

tfidf_model.fit(X_train_tfidf, y_train)

y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

tfidf_accuracy = accuracy_score(y_test, y_pred_tfidf)


print("Bag of Words Accuracy :", bow_accuracy)
print("TF-IDF Accuracy       :", tfidf_accuracy)

if bow_accuracy >= tfidf_accuracy:

    best_vectorizer = count_vectorizer
    best_model = bow_model
    best_method = "Bag of Words"
    best_accuracy = bow_accuracy

else:

    best_vectorizer = tfidf_vectorizer
    best_model = tfidf_model
    best_method = "TF-IDF"
    best_accuracy = tfidf_accuracy


print("\nBest Method:", best_method)
print("Best Accuracy:", best_accuracy)

joblib.dump(best_vectorizer, "best_vectorizer.pkl")
joblib.dump(best_model, "best_emotion_model.pkl")

print("\nBest vectorizer saved as: best_vectorizer.pkl")
print("Best model saved as: best_emotion_model.pkl")

Bag of Words Accuracy : 0.8676375326887551
TF-IDF Accuracy       : 0.7645809841414554

Best Method: Bag of Words
Best Accuracy: 0.8676375326887551

Best vectorizer saved as: best_vectorizer.pkl
Best model saved as: best_emotion_model.pkl
